# System to Attention

改寫 s2a 加入batch 以及回傳token  

## 從變異法(run_gemini)的輸出prompt，並比較他們的差異  
GPT-4o
Prompt1 表現比較好的組別  
Prompt2 表現比較差的組別  

語意上的差異  
措辭和語氣：  
Prompts 1 使用的措辭較為正式，重視對投資者預測和股價上漲的分析。  
Prompts 2 措辭相對更為謹慎和細緻，強調對基礎因素和假設的釐清。  

重點關注：  
Prompts 1 更加側重於投資者預期和股價增長的關係，主要關注「如何」投資者的預期能夠推動股價上漲。  
Prompts 2 更注重「為什麼」這些預期成立，強調對基礎驅動因素的深入了解和釐清。  

分析深度：  
Prompts 1 偏向於直接的影響和結果，強調對業務計劃和投資者預測的評估。  
Prompts 2 更深入地探討基礎驅動因素，強調透徹的分析和理解。  

作法上的差異  
分析方法：  
Prompts 1 使用較為直接的分析方法，側重於投資者預期和市場影響的關聯。  
Prompts 2 採取更加謹慎和深入的方法，強調對假設和基礎因素的驗證和闡明。  

重點範圍：  
Prompts 1 主要集中在投資者預期和市場反應，可能更適合於短期和直接的市場分析。  
Prompts 2 涵蓋更廣泛的範圍，包括對長期增長潛力和支持因素的詳細分析，適合於長期和深度的研究。  

語境和背景：  
Prompts 1 更加關注於投資者和市場的即時反應。  
Prompts 2 強調背景因素和深層次的驅動力，適合於構建更全面和透徹的分析框架。  

對語言模型的影響  
輸出內容的具體性：  
Prompts 1 可能會導致模型產生更為直接和結果導向的輸出。  
Prompts 2 會引導模型生成更為詳細和背景豐富的內容。  

分析的深度和廣度：  
Prompts 1 傾向於產生較為簡潔和重點明確的分析結果。  
Prompts 2 會引導模型進行更深入和廣泛的探討，涵蓋更多背景和基礎因素。  

結論的可靠性：  
Prompts 1 的結果可能較為直接，但在基礎和假設方面的細節可能不夠豐富。  
Prompts 2 的結果雖然可能更為複雜，但在細節和基礎方面會更為可靠和詳盡。  

總結來說，這兩組提示分別適合不同的分析需求和情境。Prompts 1 更適合於快速和直接的市場反應分析，而 Prompts 2 則適合於深度和長期的戰略研究。根據具體的分析目標選擇合適的提示組，可以更好地滿足不同的研究和分析需求。

## 範例
由GPT-4產生  
Prompts 1  
"全面評估一項業務創新計畫的可行性，並探討其對目標市場的影響。"  
"仔細驗證新業務計畫的各個部分，並預測其對公司股價的可能影響。"  
"深入探討業務創新計畫如何影響投資者的預期，並全面評估其長期效益。"  

Prompts 2  
"評估一項業務創新計畫如何改變目標市場的格局，並審慎評量其實施的風險。"   
"分析新業務計畫對公司股價的可能影響，並審慎評量其對投資者的吸引力。"  
"探討業務創新計畫如何影響投資者的預期回報，並審慎評量其對公司財務狀況的影響。"  

In [ ]:
# %pip install langchain_google_genai
# %pip install mplfinance

In [ ]:
import os
import textwrap
import math
import pandas as pd
import numpy as np

from datetime import datetime
from dateutil.relativedelta import relativedelta

import mplfinance as mpf
import google.generativeai as genai
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory

from IPython.display import display
from IPython.display import Markdown

model = 'gemini-1.5-flash'
temperature = 1
top_p = 0.9
seed = 0

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

llm = ChatGoogleGenerativeAI(
                google_api_key = os.getenv('GEMINI_API_KEY'),
                model=model,
                temperature=temperature,
                top_p=top_p,
                seed=seed,
                convert_system_message_to_human=True,
                safety_settings={
                        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
            })

Step 1 讓模型針對輸入的資訊回傳  
Step 2 讓模型對新聞評分  

In [ ]:
import time
import os
import json

def step1(stock_id, st, et):
    folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
    sorted_news_files = sorted(os.listdir(folder_path))
    sorted_news_files = [file for file in sorted_news_files if st <= datetime.strptime(file, '%Y%m%d') <= et]
    file_id = 0
    batch_size = 15

    while file_id < len(sorted_news_files):
        batch = []
        for i in range(batch_size):

            if file_id + i >= len(sorted_news_files):
                break
            with open(folder_path + sorted_news_files[file_id+i], 'r', encoding='utf-8') as f:
                news = f.read()

            # 設置初始消息列表，包括 SystemMessage 和第一個 HumanMessage
            messages = [
                ("system","你是一個厲害的股市操盤手，專長是當日沖消"),
                ("human",
                """審慎評量某企業正在實施的業務創新計劃，這些計劃包括但不限於新產品的研發、市場擴展策略、數位化轉型等。仔細審查這些創新措施對目標市場可能產生的影響，重點考慮市場需求的變化、競爭對手的反應、消費者行為的轉變以及政策法規的影響。深入分析這些創新計劃是否有助於企業提高市場佔有率，鞏固市場地位，並實現可持續發展。探討投資者對於這些創新計劃的成長預測，評估這些預測如何奠定股票的持續增長。釐清支持這些假設的根本驅動因素，包括經濟環境、行業趨勢、企業內部的研發能力、管理層的執行力等。最終提供一份詳細而精確的評估報告，幫助企業管理層和投資者做出明智的決策，確保業務創新計劃能夠帶來預期的收益和長期的競爭優勢。
                """
                 + news)
            ]
            batch.append(messages)  

        res_s2a = llm.batch(batch)

        out_path = os.path.abspath(os.getcwd()) + "/out_S2A2/step1/" + stock_id + "/"
        os.makedirs(out_path, exist_ok=True)
        for i in range(len(res_s2a)):
            output_data = {
                "model": model,
                "temperature": temperature,
                "top_p": top_p,
                "seed": seed,
                "token": res_s2a[i].usage_metadata,
                "raw_data": res_s2a[i].content,
            }

            json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
            if not os.path.exists(folder_path):
                os.makedirs(folder_path)
            with open(out_path + sorted_news_files[file_id+i] + ".json" , "w", encoding="UTF-8") as f:
                f.write(json_content)
        break
        time.sleep(60)
        file_id += batch_size


def step2(stock_id, st, et):
    folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
    sorted_news_files = sorted(os.listdir(folder_path))
    sorted_news_files = [file for file in sorted_news_files if st <= datetime.strptime(file, '%Y%m%d') <= et]
    file_id = 0
    batch_size = 15

    while file_id < len(sorted_news_files):
        batch = []
        for i in range(batch_size):

            if file_id + i >= len(sorted_news_files):
                break
            with open(folder_path + sorted_news_files[file_id+i], 'r', encoding='utf-8') as f:
                news = f.read()

            out_path = os.path.abspath(os.getcwd()) + "/out_S2A2/step1/" + stock_id + "/"
            with open(out_path + sorted_news_files[file_id+i] + ".json", 'r', encoding='utf-8') as f:
                output_data = json.load(f)
                step1_out = output_data['raw_data']

            
            # 設置初始消息列表，包括 SystemMessage 和第一個 HumanMessage
            messages = [
                ("system","你是一個厲害的股市操盤手，專長是當日沖消"),
                ("human",
                "評估一項業務創新計畫如何改變目標市場的格局，並審慎評量其實施的風險，並保留相關的資訊。" + news),
                ("assistant", step1_out),
                ("human", "請用上述資訊對於明天開盤時買進收盤時賣出的獲利可能性進行判斷，如果可以獲利請只回答#yes，否則只回答#no，如果與股市漲跌無關則回答#unknown，請盡量不要回答 #unknown"),
            ]
            batch.append(messages)
            # print(messages)

        res_s2a = llm.batch(batch)
        print(res_s2a)
        out_path = os.path.abspath(os.getcwd()) + "/out_S2A2/step2/" + stock_id + "/"
        os.makedirs(out_path, exist_ok=True)
        for i in range(len(res_s2a)):
            output_data = {
                "model": model,
                "temperature": temperature,
                "top_p": top_p,
                "seed": seed,
                "token": res_s2a[i].usage_metadata,
                "raw_data": res_s2a[i].content,
            }

            json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
            if not os.path.exists(folder_path):
                os.makedirs(folder_path)
            with open(out_path + sorted_news_files[file_id+i] + ".json" , "w", encoding="UTF-8") as f:
                f.write(json_content)
        break
        time.sleep(60)
        file_id += batch_size

In [ ]:
# 上市公司股票代號
# 代號 ['2618', '2881', '2880', '2211', '2731', '2707', '3023', '9941', '2308', '1514', '1101', '2542', '3036', '2385', '1303', '2912', '2323', '2603', '1419', '1604', '2486', '5269', '1314', '8033', '3017', '2903', '9943', '2368', '9904', '1102', '3231', '3004', '3035', '6176', '1103', '2609', '2002']
# 對應名稱 ['長榮航', '富邦金', '華南金', '長榮鋼', '雄獅', '晶華', '信邦', '裕融', '台達電', '亞力', '台泥', '興富發', '文曄', '群光', '南亞', '統一超', '中環', '長榮', '新光金', '聲寶', '一詮', '祥碩', '中石化', '遠雄', '奇鋐', '遠百', '好樂迪', '金像電', '寶成', '亞泥', '緯創', '豐達科', '智原', '瑞儀', '嘉泥', '陽明', '中鋼']
from datetime import datetime

start_day = datetime(2023, 1, 1)
end_day = datetime(2023, 12, 31)

stock_ids = ['2880']
# stock_names = ['華南金']

# stock_ids = ['2880', '2211', '2707']
# stock_names = ['華南金', '長榮鋼', '晶華']

# stock_ids = ['2618', '2881', '2880', '2211', '2707']
# stock_names = ['長榮航', '富邦金', '華南金', '長榮鋼', '晶華']

# stock_ids = ['9941', '2308', '1514', '1101', '2542']
# stock_names = ['裕融', '台達電', '亞力', '台泥', '興富發']

# run step 1
for idx in range(len(stock_ids)):
    step1(stock_ids[idx], start_day, end_day)
    print(f"step1 run stock {stock_ids[idx]} done")

time.sleep(60)
# run step 2
for idx in range(len(stock_ids)):
    step2(stock_ids[idx], start_day, end_day)
    print(f"step2 run stock {stock_ids[idx]} done")